In [1]:
import os 
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' 
os.environ['CUDA_VISIBLE_DEVICES'] = '-1' 

In [2]:
import sys 
sys.setrecursionlimit(1500) 

import gymnasium as gym
import numpy as np 

# Create the environment 
env = gym.make('CartPole-v1') 

# Set random seed for reproducibility 
np.random.seed(42) 
env.action_space.seed(42) 
env.observation_space.seed(42)

42

In [3]:
# Suppress warnings for a cleaner notebook or console experience
import warnings
warnings.filterwarnings('ignore')

# Override the default warning function
def warn(*args, **kwargs):
    pass
warnings.warn = warn

# Import necessary libraries for the Q-Learning model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input  # Import Input layer
from tensorflow.keras.optimizers import Adam
import gymnasium as gym  # Ensure the environment library is available

# Define the model building function
def build_model(state_size, action_size): 
    model = Sequential() 
    model.add(Input(shape=(state_size,)))  # Use Input layer to specify the input shape 
    model.add(Dense(24, activation='relu')) 
    model.add(Dense(24, activation='relu')) 
    model.add(Dense(action_size, activation='linear')) 
    model.compile(loss='mse', optimizer=Adam(learning_rate=0.001)) 
    return model 

# Create the environment and set up the model
env = gym.make('CartPole-v1')
state_size = int(env.observation_space.shape[0])
action_size = int(env.action_space.n)   # FIX: np.int64 -> int
model = build_model(state_size, action_size)



In [4]:
import random
import numpy as np
from collections import deque
import tensorflow as tf

# Define epsilon and epsilon_decay
epsilon = 1.0  # Starting with a high exploration rate
epsilon_min = 0.01  # Minimum exploration rate
epsilon_decay = 0.99  # Faster decay rate for epsilon after each episode

# Replay memory
memory = deque(maxlen=2000)

def remember(state, action, reward, next_state, done):
    """Store experience in memory."""
    memory.append((state, action, reward, next_state, done))

def replay(batch_size=64):  # Increased batch size
    """Train the model using a random sample of experiences from memory."""
    if len(memory) < batch_size:
        return  # Skip replay if there's not enough experience

    minibatch = random.sample(memory, batch_size)  # Sample a random batch from memory
    
    # Extract information for batch processing
    states = np.vstack([x[0] for x in minibatch])
    actions = np.array([x[1] for x in minibatch])
    rewards = np.array([x[2] for x in minibatch])
    next_states = np.vstack([x[3] for x in minibatch])
    dones = np.array([x[4] for x in minibatch])
    
    # Predict Q-values for the next states in batch
    q_next = model.predict(next_states)
    # Predict Q-values for the current states in batch
    q_target = model.predict(states)
    
    # Vectorized update of target values
    for i in range(batch_size):
        target = rewards[i]
        if not dones[i]:
            target += 0.95 * np.amax(q_next[i])  # Update Q value with the discounted future reward
        q_target[i][actions[i]] = target  # Update only the taken action's Q value
    
    # Train the model with the updated targets in batch
    model.fit(states, q_target, epochs=1, verbose=0)  # Train in batch mode

    # Reduce exploration rate (epsilon) after each training step
    global epsilon
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

def act(state):
    """Choose an action based on the current state and exploration rate."""
    if np.random.rand() <= epsilon:
        return random.randrange(action_size)  # Explore: choose a random action
    act_values = model.predict(state)  # Exploit: predict action based on the state
    return np.argmax(act_values[0])  # Return the action with the highest Q-value

# Define the number of episodes you want to train the model for
episodes = 10  # You can set this to any number you prefer
train_frequency = 5  # Train the model every 5 steps

for e in range(episodes):
    state, _ = env.reset()  # Unpack the tuple returned by env.reset()
    state = np.reshape(state, [1, state_size])
    for time in range(200):  # Limit to 200 time steps per episode
        action = act(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        reward = reward if not done else -10
        next_state = np.reshape(next_state, [1, state_size])
        remember(state, action, reward, next_state, done)  # Store experience
        state = next_state
        
        if done:
            print(f"episode: {e+1}/{episodes}, score: {time}, e: {epsilon:.2}")
            break
        
        # Train the model every 'train_frequency' steps
        if time % train_frequency == 0:
            replay(batch_size=64)  # Call replay with larger batch size for efficiency

env.close()


episode: 1/10, score: 13, e: 1.0
episode: 2/10, score: 22, e: 1.0
episode: 3/10, score: 14, e: 1.0
episode: 4/10, score: 11, e: 1.0
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
episode: 5/10, score: 11, e: 0.97
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
episode: 6/10, score: 29, e: 0.91
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━

In [5]:
for e in range(10):  

    state, _ = env.reset()  # Unpack the state from the tuple 
    state = np.reshape(state, [1, state_size])  # Reshape the state correctly 
    for time in range(500):  
        env.render()  
        action = np.argmax(model.predict(state)[0])  
        next_state, reward, terminated, truncated, _ = env.step(action)  # Unpack the five return values 
        done = terminated or truncated  # Check if the episode is done 
        next_state = np.reshape(next_state, [1, state_size])  
        state = next_state  
        if done:  
            print(f"episode: {e+1}/10, score: {time}")  
            break  

env.close() 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
episode: 1/10, score: 9
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
episode: 2/10, score: 9
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 

In [6]:
# Write your code here
%pip install gymnasium

import random
import numpy as np
import gymnasium as gym
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from collections import deque

# Reproducibility: seed Python, NumPy and TensorFlow
random.seed(42); np.random.seed(42); tf.random.set_seed(42)

# Initialize the environment
env = gym.make('CartPole-v1')

# Global settings
episodes = 100
batch_size = 32
memory = deque(maxlen=2000)
gamma = 0.95
epsilon, epsilon_min, epsilon_decay = 1.0, 0.01, 0.95

state_size = int(env.observation_space.shape[0])
action_size = int(env.action_space.n)  # FIX: np.int64 -> int

# Define the model: 3 hidden layers x 64 neurons
def build_model(state_size, action_size):
    model = Sequential()
    model.add(Input(shape=(state_size,)))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(action_size, activation='linear'))
    model.compile(loss='mse', optimizer=Adam(learning_rate=0.001))
    return model

model = build_model(state_size, action_size)

# Epsilon-greedy action selection
def act(state):
    if np.random.rand() <= epsilon:
        return random.randrange(action_size)
    q_values = model(state, training=False).numpy()
    return int(np.argmax(q_values[0]))

def remember(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

def replay(batch_size):
    minibatch = random.sample(memory, batch_size)
    states = np.vstack([s[0] for s in minibatch])
    next_states = np.vstack([s[3] for s in minibatch])
    targets = model.predict(states, verbose=0)
    target_next = model.predict(next_states, verbose=0)

    for i, (_, action, reward, _, done) in enumerate(minibatch):
        targets[i][action] = reward if done else reward + gamma * np.amax(target_next[i])

    model.fit(states, targets, epochs=1, verbose=0)

# Train
scores = []
for e in range(episodes):
    state, _ = env.reset(seed=42 if e == 0 else None)
    state = np.reshape(state, [1, state_size])
    for time in range(200):
        action = act(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        reward = reward if not done else -10
        next_state = np.reshape(next_state, [1, state_size])
        remember(state, action, reward, next_state, done)
        state = next_state

        if done:
            break
        if len(memory) > batch_size and time % 10 == 0:
            replay(batch_size)

    scores.append(time)
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay
    print(f"episode: {e+1}/{episodes}, score: {time}, e: {epsilon:.2f}")

print(f"\nAverage score over {episodes} episodes: {np.mean(scores):.1f}")
env.close()

Note: you may need to restart the kernel to use updated packages.
episode: 1/100, score: 10, e: 0.95
episode: 2/100, score: 21, e: 0.90
episode: 3/100, score: 16, e: 0.86
episode: 4/100, score: 22, e: 0.81
episode: 5/100, score: 17, e: 0.77
episode: 6/100, score: 12, e: 0.74
episode: 7/100, score: 15, e: 0.70
episode: 8/100, score: 19, e: 0.66
episode: 9/100, score: 38, e: 0.63
episode: 10/100, score: 13, e: 0.60
episode: 11/100, score: 8, e: 0.57
episode: 12/100, score: 9, e: 0.54
episode: 13/100, score: 9, e: 0.51
episode: 14/100, score: 8, e: 0.49
episode: 15/100, score: 11, e: 0.46
episode: 16/100, score: 15, e: 0.44
episode: 17/100, score: 10, e: 0.42
episode: 18/100, score: 9, e: 0.40
episode: 19/100, score: 11, e: 0.38
episode: 20/100, score: 12, e: 0.36
episode: 21/100, score: 8, e: 0.34
episode: 22/100, score: 9, e: 0.32
episode: 23/100, score: 9, e: 0.31
episode: 24/100, score: 18, e: 0.29
episode: 25/100, score: 13, e: 0.28
episode: 26/100, score: 19, e: 0.26
episode: 27/100

In [7]:
import random
import numpy as np
import gymnasium as gym
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from collections import deque

# Reproducibility: seed Python, NumPy and TensorFlow
random.seed(42); np.random.seed(42); tf.random.set_seed(42)

# Initialize the environment
env = gym.make('CartPole-v1')

# Global settings
episodes = 100
batch_size = 32
memory = deque(maxlen=2000)
gamma = 0.95
epsilon, epsilon_min, epsilon_decay = 1.0, 0.01, 0.95

state_size = int(env.observation_space.shape[0])
action_size = int(env.action_space.n)  # FIX: np.int64 -> int

# Define the model: 3 hidden layers x 64 neurons
def build_model(state_size, action_size):
    model = Sequential()
    model.add(Input(shape=(state_size,)))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(action_size, activation='linear'))
    model.compile(loss='mse', optimizer=Adam(learning_rate=0.001))
    return model

model = build_model(state_size, action_size)

# Epsilon-greedy action selection
def act(state):
    if np.random.rand() <= epsilon:
        return random.randrange(action_size)
    q_values = model(state, training=False).numpy()
    return int(np.argmax(q_values[0]))

def remember(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

def replay(batch_size):
    minibatch = random.sample(memory, batch_size)
    states = np.vstack([s[0] for s in minibatch])
    next_states = np.vstack([s[3] for s in minibatch])
    targets = model.predict(states, verbose=0)
    target_next = model.predict(next_states, verbose=0)

    for i, (_, action, reward, _, done) in enumerate(minibatch):
        targets[i][action] = reward if done else reward + gamma * np.amax(target_next[i])

    model.fit(states, targets, epochs=1, verbose=0)

# Train
scores = []
for e in range(episodes):
    state, _ = env.reset(seed=42 if e == 0 else None)
    state = np.reshape(state, [1, state_size])
    for time in range(200):
        action = act(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        reward = reward if not done else -10
        next_state = np.reshape(next_state, [1, state_size])
        remember(state, action, reward, next_state, done)
        state = next_state

        if done:
            break
        if len(memory) > batch_size and time % 10 == 0:
            replay(batch_size)

    scores.append(time)
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay
    print(f"episode: {e+1}/{episodes}, score: {time}, e: {epsilon:.2f}")

print(f"\nAverage score over {episodes} episodes: {np.mean(scores):.1f}")
env.close()

episode: 1/100, score: 10, e: 0.95
episode: 2/100, score: 21, e: 0.90
episode: 3/100, score: 16, e: 0.86
episode: 4/100, score: 22, e: 0.81
episode: 5/100, score: 17, e: 0.77
episode: 6/100, score: 12, e: 0.74
episode: 7/100, score: 15, e: 0.70
episode: 8/100, score: 19, e: 0.66
episode: 9/100, score: 38, e: 0.63
episode: 10/100, score: 13, e: 0.60
episode: 11/100, score: 8, e: 0.57
episode: 12/100, score: 9, e: 0.54
episode: 13/100, score: 9, e: 0.51
episode: 14/100, score: 8, e: 0.49
episode: 15/100, score: 11, e: 0.46
episode: 16/100, score: 15, e: 0.44
episode: 17/100, score: 10, e: 0.42
episode: 18/100, score: 9, e: 0.40
episode: 19/100, score: 11, e: 0.38
episode: 20/100, score: 12, e: 0.36
episode: 21/100, score: 8, e: 0.34
episode: 22/100, score: 9, e: 0.32
episode: 23/100, score: 9, e: 0.31
episode: 24/100, score: 18, e: 0.29
episode: 25/100, score: 13, e: 0.28
episode: 26/100, score: 19, e: 0.26
episode: 27/100, score: 11, e: 0.25
episode: 28/100, score: 11, e: 0.24
episode: 

In [8]:
def adjust_epsilon(score, consecutive_success_threshold=200):
    global epsilon 

    if score >= consecutive_success_threshold: 
        epsilon = max(epsilon_min, epsilon * 0.9)  # Reduce epsilon faster if performance is good
    else: 
        epsilon = max(epsilon_min, epsilon * epsilon_decay)  # Regular epsilon decay

episodes = 2  # Set number of episodes for training

# Train the model with adaptive epsilon decay
for e in range(episodes): 
    state = env.reset()  
    state = state[0]  # Extract the first element, which is the actual state array
    state = np.reshape(state, [1, len(state)])  # Reshape state to match the expected input shape

    total_reward = 0 

    for time in range(500):  # Limit the episode to 500 time steps
        action = act(state)  # Choose action based on policy
        next_state, reward, done, truncated, _ = env.step(action)  # Unpack 5 values

        reward = reward if not done else -10  # Penalize for reaching a terminal state
        total_reward += reward  # Accumulate rewards

        next_state = np.reshape(next_state, [1, len(next_state)])  # Reshape next state (optional based on model needs)

        remember(state, action, reward, next_state, done)  # Store experience in memory
        state = next_state  # Update the current state

        if done or truncated:  # Check if the episode is done or truncated
            adjust_epsilon(total_reward)  # Adjust epsilon based on the total reward
            print(f"episode: {e}/{episodes}, score: {time}, e: {epsilon:.2}")  # Print the episode details
            break  # Break out of the loop if the episode is done or truncated

        if len(memory) > batch_size:  # Check if enough experiences are stored in memory
            replay(batch_size)  # Train the model with the stored experiences (pass batch_size here)

episode: 0/2, score: 14, e: 0.01
episode: 1/2, score: 51, e: 0.01


In [9]:
def custom_reward(state):
    # Extract state variables: x (cart position), x_dot (cart velocity), theta (pole angle), theta_dot (pole angular velocity)
    x, x_dot, theta, theta_dot = state
    
    # Custom reward function: Encourage the agent to keep the cart near the center and the pole upright
    reward = (1 - abs(x) / 2.4) + (1 - abs(theta) / 0.20948)
    
    return reward

episodes = 2  # Number of episodes to run

# Train the model with the custom reward function
for e in range(episodes): 
    state = env.reset()  # Reset the environment

    # Print the state structure for debugging
    print(f"State: {state}, State Type: {type(state)}")

    # Extract the state if it's a tuple and reshape if necessary
    if isinstance(state, tuple):
        state = state[0]  # Extract the first element if it's a tuple

    state = np.reshape(state, [1, state_size])  # Reshape state to match the expected input shape

    for time in range(500):  # Limit the episode to 500 time steps
        action = act(state)  # Choose an action based on the current state
        
        # Unpack 5 values returned by env.step(action)
        next_state, reward, done, truncated, _ = env.step(action)

        # Compute the custom reward based on the next state
        reward = custom_reward(next_state) if not done else -10

        # Reshape next_state if necessary
        if isinstance(next_state, tuple):
            next_state = next_state[0]  # Extract the first element if it's a tuple

        next_state = np.reshape(next_state, [1, state_size])  # Reshape next state to match input shape

        # Store the experience in memory
        remember(state, action, reward, next_state, done)
        state = next_state  # Update the current state

        if done or truncated:  # If the episode is done, break out of the loop
            print(f"episode: {e}/{episodes}, score: {time}, e: {epsilon:.2}")
            break

        if len(memory) > batch_size:  # If there are enough samples in memory, train the model
            replay(batch_size)  # Train the model with a batch of experiences

State: (array([-0.03935805,  0.04991047,  0.01656847,  0.0150125 ], dtype=float32), {}), State Type: <class 'tuple'>
episode: 0/2, score: 96, e: 0.01
State: (array([-0.04095593,  0.03970334, -0.04710005, -0.02591719], dtype=float32), {}), State Type: <class 'tuple'>
episode: 1/2, score: 54, e: 0.01
